# Шаг 6. Аналитика

Превращаем тысячи строк продаж в ответы на вопросы бизнеса. Функции из `app/analytics.py`.

In [ ]:
# Подготовка: чтобы ноутбук видел папку app и подхватывал изменения в коде без перезапуска
import sys

sys.path.append("..")

%load_ext autoreload
%autoreload 2

In [ ]:
import json

from app.analytics import by_period
from app.config import SYNTHETIC_DIR
from app.pipeline import prepare_events


def read_mapping(file_name: str) -> dict[str, str]:
    # Эталон разметки колонок из .json рядом с тестовым файлом
    return json.loads((SYNTHETIC_DIR / file_name).with_suffix(".json").read_text(encoding="utf-8"))


# Готовые события трёх бизнесов - одной командой каждый
shop, _ = prepare_events(SYNTHETIC_DIR / "shop.xlsx", read_mapping("shop.xlsx"))
cafe, _ = prepare_events(SYNTHETIC_DIR / "cafe.csv", read_mapping("cafe.csv"))
salon, _ = prepare_events(SYNTHETIC_DIR / "salon.xlsx", read_mapping("salon.xlsx"))
print(len(shop), len(cafe), len(salon))

## 1. by_period: итоги по периодам

In [ ]:
# Выручка магазина по месяцам: 2 года -> 24 числа вместо 3650 строк
monthly = by_period(shop, "месяц")
monthly.round(2)

In [ ]:
# Быстрый график, чтобы увидеть то, что заложено в данные:
# рост от года к году и сезонность (пик весной-летом, спад осенью-зимой)
monthly.plot(title="Выручка магазина по месяцам", marker="o", grid=True);

In [ ]:
# По дням - видны всплески в праздники (31 декабря, 8 марта...) и выходные
by_period(shop, "день").plot(title="Выручка магазина по дням", figsize=(12, 4), grid=True);

In [ ]:
# Не только деньги: количество проданных штук по неделям в кафе
by_period(cafe, "неделя", value="quantity").head()

In [ ]:
# Выручка всех трёх бизнесов по годам - одна и та же функция.
# ОСТОРОЖНО: годы неполные! В 2024 только 3 месяца (с октября), в 2026 - 9 (по сентябрь).
# Поэтому кажется, что 2026 хуже 2025, хотя на самом деле продажи растут.
# Сравнивать можно только одинаковые по длине периоды - этим займётся compare_periods
for name, events in [("магазин", shop), ("кафе", cafe), ("салон", salon)]:
    yearly = by_period(events, "год").round(2)
    print(name, dict(zip(yearly.index.year, yearly)))

### Попробуй сломать

In [ ]:
# У салона нет количества - понятная ошибка вместо столбца нулей
by_period(salon, "месяц", value="quantity")

In [ ]:
# Опечатка в периоде
by_period(shop, "квартал")